In [ ]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

import numpy as np

from tokenizers import ByteLevelBPETokenizer

from transformers import PreTrainedTokenizerFast

from sklearn.model_selection import train_test_split

from imblearn.over_sampling import RandomOverSampler

In [ ]:
class EHR(Dataset):
    def __init__(
            self,
            df                      : pd.DataFrame,
            labels                  : pd.Series,
            tabular_features        : list,
            unstructured_features   : list,
            tokenizer               : object,
            tokens_len              : int,
            scaler                  : StandardScaler
        ):
        
        # assert if there are any tabular features, that they are all numerical, otherwise raise error
        for feature in tabular_features:
            if not pd.api.types.is_numeric_dtype(df[feature]):
                raise ValueError(f"The column '{feature}' doesn't have numerical data.")
        
        dataset_tabular = df[tabular_features].astype('float32')
        dataset_unstructured = df[unstructured_features]
        
        if not hasattr(scaler, 'scale_'):
            #print('ENTRATO')
            dataset_tabular[tabular_features] = scaler.fit_transform(dataset_tabular)
        else:
            dataset_tabular[tabular_features] = scaler.transform(dataset_tabular)
        
        for feature in unstructured_features:
            tokens = tokenizer(
                list(dataset_unstructured[feature]),
                truncation = True, 
                padding = 'max_length', 
                max_length = tokens_len,
                return_tensors = 'np'
            )
            
            #print(tokens['input_ids'].tolist())
            
            dataset_unstructured.loc[:, feature] = tokens['input_ids'].tolist()
        
        self.dataset_tabular = dataset_tabular
        self.dataset_unstructured = dataset_unstructured
        self.labels = labels
        self.masks = tokens['attention_mask'].tolist()
    
    def __getitem__(self, index):
        return torch.tensor(self.dataset_tabular.iloc[[index]].values, dtype=torch.float32).squeeze(), \
            torch.tensor(self.dataset_unstructured.iloc[[index]].values[0][0], dtype=torch.long), \
            torch.tensor(self.masks[index], dtype=torch.bool), \
            torch.tensor(self.labels.iloc[index], dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)

In [ ]:
df = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_consults_medication.parquet')

labels = df.iloc[:, -1]
df = df.iloc[:, 1:-1]

desired_percentage = 0

class_counts = labels.value_counts()

majority_class = class_counts.max()

alpha = (desired_percentage * len(labels)) / majority_class

if desired_percentage:
    oversampler = RandomOverSampler(sampling_strategy = alpha,
        random_state=42)

else:
    oversampler = None

tabular_features = df.select_dtypes(include=['number']).columns.tolist()

unstructured_features = df.select_dtypes(include=['object']).columns.tolist()

indices = np.arange(df.shape[0])

trainval_indices, test_indices = train_test_split(
    indices,
    test_size=0.2, 
    stratify=labels, 
    random_state=42
)

In [ ]:
tokenizer_vocab_path  = r'C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\ODIN\sandboxes\pretrained\DutchEHRTokenizer\vocab.json'
tokenizer_merges_path = r'C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\ODIN\sandboxes\pretrained\DutchEHRTokenizer\merges.txt'

tokenizer = ByteLevelBPETokenizer(
    vocab = tokenizer_vocab_path,
    merges = tokenizer_merges_path,
    add_prefix_space = True
)

tokenizer = PreTrainedTokenizerFast(tokenizer_object = tokenizer._tokenizer)
tokenizer.add_special_tokens({'pad_token': '<pad>'})

tokens_len = 4096

scaler = StandardScaler()

df_test = df.iloc[test_indices]
labels_test = labels.iloc[test_indices]

# Viene caricato il dataset di train
dataset_test = EHR(
    df = df_test,
    labels = labels_test,
    tabular_features = tabular_features,
    unstructured_features = unstructured_features,
    scaler = scaler,
    tokenizer = tokenizer,
    tokens_len = tokens_len
)

In [ ]:
data, tokens, masks, targets = dataset_test.__getitem__(0)

In [ ]:
train_loader = DataLoader(
    dataset_test,
    batch_size = 10,
    shuffle = False,
    pin_memory = True
)


for data, tokens, masks, targets in train_loader:
    
    print(data.shape, tokens.shape, targets.shape)
    
    break

In [ ]:
skf = StratifiedKFold(
    n_splits = 5,
    shuffle=True,
    random_state=42
)

scaler = StandardScaler()

df_trainval = df.iloc[trainval_indices]
labels_trainval = labels[trainval_indices]

for split, (train_indices, val_indices) in enumerate(skf.split(df_trainval, labels_trainval)):
    
    df_train = df_trainval.iloc[train_indices]
    labels_train = labels_trainval.iloc[train_indices]
    
    if oversampler:
        df_train, labels_train = oversampler.fit_resample(df_train, labels_train)
    
    dataset_train = EHR(
        df = df_train,
        labels = labels_train,
        tokens_len = tokens_len,
        tabular_features = tabular_features,
        scaler = scaler,
        unstructured_features = unstructured_features,
        tokenizer = tokenizer
    )
    
    break

labels_train

# Conta le occorrenze di ciascuna classe
class_counts = labels_train.value_counts()

# Calcola la percentuale di ciascuna classe
class_percentages = (class_counts / len(labels_train)) * 100

print("Percentuali delle classi:")
print(class_percentages)

# Dataset analysis

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_consults_medication.parquet')

df


In [ ]:
print(df['age'].mean(), df['age'].std())